# Clickhouse

Clickhouse is an open-source OLAP DBMS. This DBMS specialises in processing large ammounts of data and is primarily used by analytics and ML teams to extract data from huge datasets.

## Describe

Use the `TABLE DESCRIBE` command to view the properties of the table. This is practcally useful for identifing the names and types of the columns.

Check more in the page [DESCRIBE TABLE](https://clickhouse.com/docs/sql-reference/statements/describe-table).

---

The following cell shows the example of `DESCRIBE TABLE` usage.

In [3]:
--ClickHouse
DESCRIBE TABLE
SELECT *
FROM VALUES(
    'id UInt32, name String',
    (1, 'Alice'),
    (2, 'Bob'),
    (3, 'Carol')
);

name,type,default_type,default_expression,comment,codec_expression,ttl_expression
id,UInt32,,,,,
name,String,,,,,


## Databases

Most essences in clickhouse are separated by databases. The database can contain: tables, views, dictionaries and functions.

Important commands to handle databases:

- `SHOW DATABASES`.
- `CREATE DATABASE <name>`
- `DROP DATABASE <name>`.
- `USE DATABASE`: to select the context, all subsequent queries will automatilly will refer to the essesnces of the specific database.

---

The following cell show the initial list of available databases.

In [2]:
--ClickHouse
SHOW DATABASES;

name
INFORMATION_SCHEMA
default
information_schema
system


After `CREATE DATABASE` there is a new database with the corresponding name:

In [5]:
--ClickHouse
CREATE DATABASE test_database;
SHOW DATABASES;

name
INFORMATION_SCHEMA
default
information_schema
system
test_database


After `DROP DATABASE` the database is removed.

In [6]:
--ClickHouse
DROP DATABASE test_database;
SHOW DATABASES;

name
INFORMATION_SCHEMA
default
information_schema
system


## Parts

The clickhouse stores data as parts. It manipulates these parts over time. Information about the current parts is stored in the `system.parts` table.

The practically important columns for the `system.parts` table are:

- `name`: the name of the part, there is encoded some iformation about the part.
- `table`: describes to which table part is belongs to.
- `active`: colum takes value 1 if the corresponding part actually is used.

---

The following cell displays the description for the `system.parts` table.

In [2]:
--ClickHouse
DESCRIBE TABLE system.parts;

name,type,default_type,default_expression,comment,codec_expression,ttl_expression
partition,String,,,The partition name.,,
name,String,,,"Name of the data part. The part naming structure can be used to determine many aspects of the data, ingest, and merge patterns. The part naming format is the following: ``` <partition_id>_<minimum_block_number>_<maximum_block_number>_<level>_<data_version> ``` * Definitions: - `partition_id` - identifies the partition key - `minimum_block_number` - identifies the minimum block number in the part. ClickHouse always merges continuous blocks - `maximum_block_number` - identifies the maximum block number in the part - `level` - incremented by one with each additional merge on the part. A level of 0 indicates this is a new part that has not been merged. It is important to remember that all parts in ClickHouse are always immutable - `data_version` - optional value, incremented when a part is mutated (again, mutated data is always only written to a new part, since parts are immutable)",,
uuid,UUID,,,The UUID of data part.,,
part_type,String,,,The data part storing format. Possible Values: Wide (a file per column) and Compact (a single file for all columns).,,
active,UInt8,,,"Flag that indicates whether the data part is active. If a data part is active, it's used in a table. Otherwise, it's about to be deleted. Inactive data parts appear after merging and mutating operations.",,
marks,UInt64,,,"The number of marks. To get the approximate number of rows in a data part, multiply marks by the index granularity (usually 8192) (this hint does not work for adaptive granularity).",,
rows,UInt64,,,The number of rows.,,
files,UInt64,,,The number of files in the data part.,,
bytes_on_disk,UInt64,,,Total size of all the data part files in bytes.,,
data_compressed_bytes,UInt64,,,"Total size of compressed data in the data part. All the auxiliary files (for example, files with marks) are not included.",,


## Primary key

Primary key in clickhouse have differnt properties from other DBMS's.

In clichouse, the primary key does not required to have unique values, mainly used as "primary inde". The table is sorted and separted into granules accordingly to the primary keys. This is due to the architecture of clickhouse.

It is a good idea to choose the columns with low cardinality (i.e. small number of unique values) as the primary key. ClickHouse uses these columns to separte data internally data into granules, which are then used quckly query the data.

---

The following cell creates the table, and specifies `id` to be the primary key:

In [16]:
--ClickHouse
DROP TABLE IF EXISTS pk_example;
CREATE TABLE pk_example(
    id INT,
    val TEXT
)
ENGINE = MergeTree()
PRIMARY KEY (id);

read_rows,read_bytes,written_rows,written_bytes,total_rows_to_read,result_rows,result_bytes,elapsed_ns,memory_usage,query_id
0,0,0,0,0,0,0,37294218,1119215,490ec853-80ab-45c1-a0bf-267ab55f8730


Inserting some data to clickhouse:

In [17]:
--ClickHouse
INSERT INTO pk_example VALUES (2, 'some extra'), (1, 'data1'), (1, 'data2');

Note that it is ok to enter the same values for the column that is is defined as the primary key, and the data will be sorted accodingly:

In [18]:
--ClickHouse
SELECT * FROM pk_example;

id,val
1,data1
1,data2
2,some extra


### Optimization

In clichouse, the primary key sorts the data and separates it to granules. If you specify a primary key in a query filter, the engine will only check those granules that begong to that primary key value.

---

The following cell creates a data table containing fields `id` and `category`, with `category` is defined as the primary key.

In [111]:
--ClickHouse
DROP TABLE IF EXISTS optim_example;
CREATE TABLE optim_example(
    id INT,
    category String
)
PRIMARY KEY category;

INSERT INTO optim_example
SELECT
    number AS id,
    CASE
        WHEN number < 25000 THEN 'cat'
        WHEN number < 50000 THEN 'dog'
        WHEN number < 75000 THEN 'mouse'
        ELSE 'elephant'
    END
FROM numbers(100000);

read_rows,read_bytes,written_rows,written_bytes,total_rows_to_read,result_rows,result_bytes,elapsed_ns,memory_usage,query_id
0,0,0,0,0,0,0,28296947,1119159,147e2090-f262-4ca4-acc7-7a9fd99b594a


**Note**: It may be confusing that the primary key is not the `id` column, but the column that contains the some kind of category. Keep in mind that the primary key in clickhouse have different meaning.

The following table shows the distribution of the categories:

In [110]:
--ClickHouse
SELECT
    category,
    COUNT(*)
FROM optim_example
GROUP BY category;

category,COUNT()
elephant,250000
dog,250000
cat,250000
mouse,250000


Now consider how many rows are read to execute query:

```sql
select * from optim_example where id=12345;
```

```
   ┌────id─┬─category─┐
1. │ 12345 │ cat      │
   └───────┴──────────┘

1 row in set. Elapsed: 0.003 sec. Processed 100.00 thousand rows, 449.15 KB (33.43 million rows/s., 150.16 MB/s.)
Peak memory usage: 165.52 KiB.
```

**Note** `Processed 100.00 thousand rows`, which means they query needs to check every row in the table.

And query to the same table which filters by the primary key:

```sql
select * from optim_example where category='mouse';
```

```
...
25000 rows in set. Elapsed: 0.003 sec. Processed 26.27 thousand rows, 319.08 KB (8.22 million rows/s., 99.83 MB/s.)
Peak memory usage: 581.20 KiB.
```

A much lower number of rows were read by the computer - clickhouse worked only with the granules corresponding to the `category='mouse'`.

## Datatypes

Check the available ClickHouse datatypes the system table `system.data_type_family`.

Some types are just alices to the other ones.

Check the description of the different datatypes in clickhouse in the corresponding [reference page](https://clickhouse.com/docs/sql-reference/data-types).

---

The following cell shows the contents of the `system.data_type_family` table.

In [3]:
--ClickHouse
SELECT * FROM system.data_type_families LIMIT 10;

name,case_insensitive,alias_to
JSON,1,
Dynamic,0,
Geometry,0,
Polygon,0,
Ring,0,
Point,0,
SimpleAggregateFunction,0,
IntervalYear,0,
IntervalMonth,0,
IntervalWeek,0,
